# Validación Temporal — Dengue CON Signos de Alarma (DCSA)
## Robustez del modelo ante cambio de período epidémico

---

## Motivación

El notebook `03a` entrenó y evaluó el modelo con **partición aleatoria** (80/20).
Este enfoque mezcla datos de todos los años, lo que puede producir un modelo
que funciona bien *en el pasado* pero se degrada al enfrentarse a un brote nuevo.

Este notebook responde una pregunta crítica para la validez clínica:
> **¿El modelo entrenado con datos de 2015–2021 generaliza correctamente
> a los brotes de 2022–2025?**

## Contexto epidemiológico del corte temporal

El análisis de los datos revela una **ruptura estructural clara** en 2021–2022:

| Período | Tasa hosp. DCSA | Interpretación |
|---------|----------------|----------------|
| 2015–2020 | 19.7% – 43.8% | Período heterogéneo, posible cambio de protocolo INS |
| 2021 | 83.1% | Inflexión — nueva era de clasificación |
| 2022–2025 | 85.8% – 89.1% | Régimen estable de alta hospitalización |

Esta ruptura no es clínica sino metodológica: desde 2021 el INS aplicó
criterios más estrictos de clasificación DCSA, resultando en que casi todos
los casos clasificados como DCSA terminan hospitalizados.
El modelo de 03a fue entrenado mayoritariamente con el régimen 2015–2020
donde la señal clínica era más heterogénea.

**Pregunta de validación:** ¿el modelo aprendió patrones clínicos reales
o aprendió el patrón del período 2015–2020?

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# BANDERA DE EJECUCION
# FORZAR_RECALCULO = True  → recalcula todo desde cero
# FORZAR_RECALCULO = False → reutiliza checkpoints
# ═══════════════════════════════════════════════════════════════════════
FORZAR_RECALCULO = False

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
import joblib
import gc
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    f1_score, roc_auc_score, balanced_accuracy_score,
    precision_score, recall_score, average_precision_score,
    confusion_matrix, RocCurveDisplay, PrecisionRecallDisplay,
    classification_report, make_scorer)
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')
%matplotlib inline

plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 13, 'axes.titleweight': 'bold',
    'axes.labelsize': 11, 'legend.frameon': False,
})
COLORS = {
    'primary': '#2C7BB6', 'secondary': '#D7191C',
    'accent': '#FDAE61', 'neutral': '#7F7F7F',
    'con_signos': '#E74C3C', 'sin_signos': '#3498DB',
    'train': '#2C7BB6', 'test': '#D7191C',
}
SIGNOS_ALARMA      = ['DOLOR_ABDO','VOMITO','DIARREA','SOMNOLENCI','HIPOTENSIO',
                      'HEPATOMEG','HEM_MUCOSA','HIPOTERMIA','AUM_HEMATO','CAIDA_PLAQ','ACUM_LIQUI']
SINTOMAS_GENERALES = ['FIEBRE','CEFALEA','DOLRRETROO','MALGIAS','ARTRALGIA','ERUPCIONR']
TARGET             = 'TARGET_HOSPITALIZACION_ORIGINAL'
RANDOM_STATE       = 42

ROOT        = Path(r'D:\Documentos\GitHub\dengue')  # <-- edita si es necesario
OUT_FIG     = ROOT / 'outputs' / 'figures'
OUT_TAB     = ROOT / 'outputs' / 'tables'
INTERIM     = ROOT / 'data' / 'interim'
CHECKPOINTS = ROOT / 'outputs' / 'checkpoints'
for p in [OUT_FIG, OUT_TAB, INTERIM, CHECKPOINTS]:
    p.mkdir(parents=True, exist_ok=True)

def checkpoint_existe(path):
    if FORZAR_RECALCULO: return False
    return Path(path).exists()

def guardar_checkpoint(obj, path, nombre=''):
    joblib.dump(obj, path)
    mb = Path(path).stat().st_size / 1e6
    print(f'  Guardado {nombre}: {Path(path).name} ({mb:.1f} MB)')

def limpiar_memoria(*objetos):
    for obj in objetos:
        try: del obj
        except: pass
    gc.collect()

def memoria_usada():
    import psutil, os
    mb = psutil.Process(os.getpid()).memory_info().rss / 1e6
    print(f'  RAM en uso: {mb:.0f} MB')

# Cargar datos completos
df_con = joblib.load(INTERIM / 'datos_con_signos.pkl')
print(f'DCSA total: {len(df_con):,} casos')
print(f'Rango temporal: {df_con["AÑO"].min()} – {df_con["AÑO"].max()}')

## 1. Análisis del drift temporal

Antes de evaluar el modelo, es necesario cuantificar cuánto cambiaron
los datos entre el período de entrenamiento (2015–2021) y el de prueba (2022–2025).

Un **drift severo** en la prevalencia de síntomas o en la tasa de hospitalización
anticipará una degradación del modelo — y justificará estrategias de adaptación
como re-entrenamiento periódico o modelos con ventana deslizante.

In [ ]:
# Corte temporal definido por la ruptura estructural detectada en el EDA
# 2015-2021: regimen heterogeneo (tasa hosp DCSA: 19.7% - 83.1%)
# 2022-2025: regimen estable de alta hospitalizacion (85.8% - 89.1%)
ANIO_CORTE = 2022

df_train_t = df_con[df_con['AÑO'] < ANIO_CORTE].copy()
df_test_t  = df_con[df_con['AÑO'] >= ANIO_CORTE].copy()

print('Distribucion por periodo:')
print(f'  Train temporal (2015-2021): {len(df_train_t):,} casos | '
      f'Hosp: {df_train_t[TARGET].mean():.1%}')
print(f'  Test temporal  (2022-2025): {len(df_test_t):,} casos  | '
      f'Hosp: {df_test_t[TARGET].mean():.1%}')

# Distribucion anual con tasas
anio_dist = df_con.groupby('AÑO').agg(
    N=(TARGET,'size'),
    Hosp=(TARGET, lambda x: x.eq(1).sum())
).reset_index()
anio_dist['Pct_hosp'] = (anio_dist['Hosp']/anio_dist['N']*100).round(1)
anio_dist['Periodo']  = anio_dist['AÑO'].apply(
    lambda x: 'Train (2015-2021)' if x < ANIO_CORTE else 'Test (2022-2025)')
display(anio_dist)

# Grafico de tasa de hospitalizacion por anio
fig, ax = plt.subplots(figsize=(13, 5))
colores_anio = [COLORS['train'] if a < ANIO_CORTE else COLORS['test']
    for a in anio_dist['AÑO']]
bars = ax.bar(anio_dist['AÑO'].astype(str), anio_dist['Pct_hosp'],
    color=colores_anio, alpha=0.85, edgecolor='white', width=0.7)
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)

ax.axvline(4.5, color='black', linewidth=2, linestyle='--')
ax.text(2.2, ax.get_ylim()[1]*0.95, 'TRAIN\n2015–2021',
    ha='center', fontsize=10, color=COLORS['train'], fontweight='bold')
ax.text(6.5, ax.get_ylim()[1]*0.95, 'TEST\n2022–2025',
    ha='center', fontsize=10, color=COLORS['test'], fontweight='bold')

leyenda = [mpatches.Patch(color=COLORS['train'], label='Periodo train'),
           mpatches.Patch(color=COLORS['test'],  label='Periodo test')]
ax.legend(handles=leyenda, fontsize=9)
ax.set_xlabel('Año epidemiologico')
ax.set_ylabel('Pacientes hospitalizados (%)')
ax.set_title(
    'Tasa de hospitalizacion DCSA por año\n'
    'Ruptura estructural en 2021-2022: cambio de regimen de clasificacion INS')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(OUT_FIG / 'temporal_tasa_hospitalizacion.png',
    dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Drift en prevalencia de signos de alarma entre periodos
# Si los signos cambian de prevalencia, el modelo entrenado en 2015-2021
# puede no reconocer los patrones del periodo 2022-2025
todas = SINTOMAS_GENERALES + SIGNOS_ALARMA

prev_train = (df_train_t[todas].eq(1).mean() * 100).rename('Train 2015-2021')
prev_test  = (df_test_t[todas].eq(1).mean()  * 100).rename('Test 2022-2025')
drift_df   = pd.concat([prev_train, prev_test], axis=1)
drift_df['Diferencia_%'] = (drift_df['Test 2022-2025'] - drift_df['Train 2015-2021']).round(2)
drift_df['Drift_abs']    = drift_df['Diferencia_%'].abs()
drift_df = drift_df.sort_values('Drift_abs', ascending=False)

print('Drift en prevalencia de manifestaciones clinicas:')
display(drift_df.round(2))
drift_df.to_csv(OUT_TAB / 'drift_prevalencia_clinica.csv')

# Grafico de drift
fig, ax = plt.subplots(figsize=(12, 7))
x = np.arange(len(drift_df))
w = 0.35
ax.bar(x - w/2, drift_df['Train 2015-2021'], w,
    color=COLORS['train'], alpha=0.85, label='Train 2015-2021', edgecolor='white')
ax.bar(x + w/2, drift_df['Test 2022-2025'], w,
    color=COLORS['test'], alpha=0.85, label='Test 2022-2025', edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(drift_df.index, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Prevalencia (%)')
ax.set_title(
    'Drift en prevalencia de manifestaciones clinicas\n'
    'Train (2015-2021) vs Test temporal (2022-2025)')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(OUT_FIG / 'temporal_drift_prevalencia.png',
    dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## 2. Preparación de features para validación temporal

Se usan exactamente los mismos features que en `03a` para que
la comparación sea válida. El modelo de `03a` se carga directamente
desde el checkpoint — no se re-entrena.

In [ ]:
# Cargar mejor modelo de 03a
CP_MODELO_03A = CHECKPOINTS / 'mejor_modelo_DCSA.pkl'
assert CP_MODELO_03A.exists(), 'Ejecuta primero 03a_modelado_con_signos.ipynb'

saved_03a  = joblib.load(CP_MODELO_03A)
modelo_03a = saved_03a['pipeline']
mejor_03a  = saved_03a['modelo']
FEATURES   = saved_03a['features']

# Verificar que AÑO no esta en features
assert 'AÑO' not in FEATURES, 'ERROR: AÑO no debe estar en FEATURES'

print(f'Modelo cargado: {mejor_03a}')
print(f'Features ({len(FEATURES)}): {FEATURES}')
print(f'F1 en test aleatorio (03a): {saved_03a["tabla_test"].iloc[0]["F1"]:.4f}')

vars_clinicas = SINTOMAS_GENERALES + SIGNOS_ALARMA
vars_demo     = [v for v in FEATURES if v not in vars_clinicas + ['SEMANA']]

def preparar_X(df):
    """Prepara features con los mismos pasos que 03a."""
    df_m = df[FEATURES + [TARGET]].copy()
    for v in vars_clinicas:
        df_m[v] = df_m[v].map({1: 1, 2: 0})
    for v in vars_demo:
        if str(df_m[v].dtype) in ('object',) or str(df_m[v].dtype).startswith('string'):
            df_m[v] = pd.Categorical(df_m[v]).codes.astype(float)
            df_m[v] = df_m[v].replace(-1, np.nan)
    for col in FEATURES:
        df_m[col] = pd.to_numeric(df_m[col], errors='coerce').astype(np.float64)
    return df_m.dropna(subset=[TARGET])

df_train_prep = preparar_X(df_train_t)
df_test_prep  = preparar_X(df_test_t)

X_train_t = df_train_prep[FEATURES]
y_train_t = df_train_prep[TARGET].astype(int)
X_test_t  = df_test_prep[FEATURES]
y_test_t  = df_test_prep[TARGET].astype(int)

print(f'Train temporal: {len(X_train_t):,} | Hosp: {y_train_t.mean():.1%}')
print(f'Test temporal : {len(X_test_t):,}  | Hosp: {y_test_t.mean():.1%}')

## 3. Evaluación del modelo 03a sobre el test temporal

Se evalúa el modelo entrenado con partición aleatoria (03a) directamente
sobre los datos de 2022–2025, sin ningún re-entrenamiento.
Esto mide la **degradación real** del modelo ante el cambio de período.

In [ ]:
# Evaluacion del modelo 03a sobre test temporal 2022-2025
# Esta es la evaluacion de generalizacion mas critica:
# el modelo nunca vio estos datos durante el entrenamiento
y_prob_03a = modelo_03a.predict_proba(X_test_t)[:, 1]
y_pred_03a = modelo_03a.predict(X_test_t)

metricas_03a = {
    'Escenario':   'Modelo 03a sobre test temporal (2022-2025)',
    'F1':          round(f1_score(y_test_t, y_pred_03a), 4),
    'AUC_ROC':     round(roc_auc_score(y_test_t, y_prob_03a), 4),
    'AUC_PR':      round(average_precision_score(y_test_t, y_prob_03a), 4),
    'Precision':   round(precision_score(y_test_t, y_pred_03a, zero_division=0), 4),
    'Recall':      round(recall_score(y_test_t, y_pred_03a), 4),
    'Bal_Acc':     round(balanced_accuracy_score(y_test_t, y_pred_03a), 4),
}

# Metricas del modelo 03a en su test aleatorio (referencia)
metricas_aleatorio = dict(saved_03a['tabla_test'].iloc[0])
metricas_aleatorio['Escenario'] = 'Modelo 03a sobre test aleatorio (03a)'

tabla_comparativa = pd.DataFrame([metricas_aleatorio, metricas_03a])
print('Comparativa: test aleatorio vs test temporal')
display(tabla_comparativa[['Escenario','F1','AUC_ROC','AUC_PR','Precision','Recall','Bal_Acc']])

# Calcular degradacion
deg_f1  = metricas_aleatorio['F1']  - metricas_03a['F1']
deg_auc = metricas_aleatorio['AUC_ROC'] - metricas_03a['AUC_ROC']
print(f'\nDegradacion F1      : {deg_f1:+.4f} ({deg_f1/metricas_aleatorio["F1"]*100:+.1f}%)')
print(f'Degradacion AUC-ROC : {deg_auc:+.4f} ({deg_auc/metricas_aleatorio["AUC_ROC"]*100:+.1f}%)')

print('\nReporte de clasificacion — test temporal:')
print(classification_report(y_test_t, y_pred_03a,
    target_names=['No hosp.', 'Hospitalizado']))

In [ ]:
# Matriz de confusion dual: test aleatorio vs test temporal
def plot_cm_dual(y_true, y_pred, y_prob, titulo, axes):
    cm     = confusion_matrix(y_true, y_pred)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    etiq   = ['No hosp.', 'Hospitalizado']
    for ax, datos, fmt, cmap, subtit in zip(
        axes,
        [cm, cm_pct], ['{:.0f}', '{:.1f}%'],
        [plt.cm.Blues, plt.cm.Oranges],
        ['Conteos', '% por fila']
    ):
        im = ax.imshow(datos, interpolation='nearest', cmap=cmap)
        plt.colorbar(im, ax=ax, shrink=0.75)
        thresh = datos.max() / 2
        for i in range(2):
            for j in range(2):
                color = 'white' if datos[i,j] > thresh else '#222'
                ax.text(j, i, fmt.format(datos[i,j]),
                    ha='center', va='center',
                    fontsize=12, fontweight='bold', color=color)
        ax.set_xticks([0,1]); ax.set_yticks([0,1])
        ax.set_xticklabels(etiq, fontsize=9)
        ax.set_yticklabels(etiq, fontsize=9)
        ax.set_xlabel('Prediccion', fontsize=9)
        ax.set_ylabel('Real', fontsize=9)
        ax.set_title(subtit, fontsize=9, fontweight='bold')
    f1v = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)
    axes[0].set_ylabel(f'Real\nF1={f1v:.3f} | AUC={auc:.3f}', fontsize=8)

# Obtener predicciones del test aleatorio de 03a
X_test_al, y_test_al = saved_03a['tabla_test'], None
# Recargar particion aleatoria
CP_PART = CHECKPOINTS / 'particion_DCSA.pkl'
if CP_PART.exists():
    _, X_test_al, _, y_test_al, _ = joblib.load(CP_PART)
    y_pred_al = modelo_03a.predict(X_test_al)
    y_prob_al = modelo_03a.predict_proba(X_test_al)[:, 1]
    tiene_aleatorio = True
else:
    tiene_aleatorio = False
    print('Particion aleatoria no disponible — solo se muestra test temporal')

n_cols = 2 if tiene_aleatorio else 1
fig, axes_grid = plt.subplots(n_cols, 2, figsize=(12, 5 * n_cols))
if n_cols == 1:
    axes_grid = [axes_grid]

if tiene_aleatorio:
    plot_cm_dual(y_test_al, y_pred_al, y_prob_al,
        'Test aleatorio 80/20', axes_grid[0])
    fig.text(0.5, axes_grid[0][0].get_position().y1 + 0.01,
        'A. Test aleatorio (particion 03a)',
        ha='center', fontsize=11, fontweight='bold', transform=fig.transFigure)
    plot_cm_dual(y_test_t, y_pred_03a, y_prob_03a,
        'Test temporal 2022-2025', axes_grid[1])
    fig.text(0.5, axes_grid[1][0].get_position().y1 + 0.01,
        'B. Test temporal (2022-2025)',
        ha='center', fontsize=11, fontweight='bold', transform=fig.transFigure)
else:
    plot_cm_dual(y_test_t, y_pred_03a, y_prob_03a, '', axes_grid[0])

plt.suptitle(
    f'Matrices de confusion — {mejor_03a}\n'
    'Comparativa particion aleatoria vs validacion temporal',
    fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(OUT_FIG / 'temporal_confusion_comparativa.png',
    dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## 4. Re-entrenamiento con partición temporal

Se re-entrena el mismo pipeline de `03a` usando exclusivamente datos
de 2015–2021 como train y 2022–2025 como test.
Esto responde: **¿mejora el modelo si se entrena con el régimen correcto?**

> Si el modelo re-entrenado temporalmente supera significativamente al de `03a`,
> indica que la partición aleatoria introducía data leakage temporal
> (datos futuros en el train contaminaban la evaluación).

In [ ]:
import importlib
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

CP_TEMPORAL = CHECKPOINTS / 'eval_temporal_DCSA.pkl'

if checkpoint_existe(CP_TEMPORAL):
    print('Re-entrenamiento temporal ya calculado — cargando...')
    saved_temp = joblib.load(CP_TEMPORAL)
    modelo_temp   = saved_temp['pipeline']
    y_pred_temp   = saved_temp['y_pred']
    y_prob_temp   = saved_temp['y_prob']
    metricas_temp = saved_temp['metricas']
else:
    # Reconstruir pipeline identico al de 03a
    vars_clinicas_l = SINTOMAS_GENERALES + SIGNOS_ALARMA
    vars_demo_l     = [v for v in FEATURES if v not in vars_clinicas_l + ['SEMANA']]
    vars_tiempo_l   = [v for v in FEATURES if v in ['SEMANA']]
    cols_num        = vars_demo_l + vars_tiempo_l

    preprocesador = ColumnTransformer([
        ('num', Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('sc',  StandardScaler())
        ]), cols_num),
        ('bin', SimpleImputer(strategy='most_frequent'), vars_clinicas_l),
    ], remainder='drop')

    # Usar el mismo tipo de modelo que gano en 03a
    clf = ExtraTreesClassifier(n_estimators=300, max_depth=10,
        random_state=RANDOM_STATE, n_jobs=1)

    modelo_temp = ImbPipeline([
        ('prep',  preprocesador),
        ('smote', SMOTE(random_state=RANDOM_STATE, k_neighbors=5)),
        ('clf',   clf),
    ])

    print(f'Entrenando {mejor_03a} con datos 2015-2021...')
    modelo_temp.fit(X_train_t, y_train_t)
    y_prob_temp = modelo_temp.predict_proba(X_test_t)[:, 1]
    y_pred_temp = modelo_temp.predict(X_test_t)

    metricas_temp = {
        'Escenario':  'Modelo re-entrenado temporal (2015-2021 → 2022-2025)',
        'F1':         round(f1_score(y_test_t, y_pred_temp), 4),
        'AUC_ROC':    round(roc_auc_score(y_test_t, y_prob_temp), 4),
        'AUC_PR':     round(average_precision_score(y_test_t, y_prob_temp), 4),
        'Precision':  round(precision_score(y_test_t, y_pred_temp, zero_division=0), 4),
        'Recall':     round(recall_score(y_test_t, y_pred_temp), 4),
        'Bal_Acc':    round(balanced_accuracy_score(y_test_t, y_pred_temp), 4),
    }
    guardar_checkpoint({'pipeline': modelo_temp, 'y_pred': y_pred_temp,
        'y_prob': y_prob_temp, 'metricas': metricas_temp},
        CP_TEMPORAL, 're-entrenamiento temporal')
    limpiar_memoria()

print('\nMetricas modelo re-entrenado temporalmente:')
display(pd.DataFrame([metricas_temp]))
memoria_usada()

## 5. Tabla comparativa final

Se consolidan los tres escenarios de evaluación para una
interpretación integral de la robustez del modelo.

In [ ]:
# Consolidar los tres escenarios
tabla_final = pd.DataFrame([
    {
        'Escenario': 'A. Test aleatorio (03a)',
        'Train': '80% aleatorio (2015-2025)',
        'Test':  '20% aleatorio (2015-2025)',
        'F1':        round(saved_03a['tabla_test'].iloc[0]['F1'], 4),
        'AUC_ROC':   round(saved_03a['tabla_test'].iloc[0]['AUC_ROC'], 4),
        'AUC_PR':    round(saved_03a['tabla_test'].iloc[0]['AUC_PR'], 4),
        'Recall':    round(saved_03a['tabla_test'].iloc[0]['Recall'], 4),
        'Bal_Acc':   round(saved_03a['tabla_test'].iloc[0]['Bal_Acc'], 4),
    },
    {
        'Escenario': 'B. Test temporal — modelo 03a',
        'Train': '80% aleatorio (2015-2025)',
        'Test':  '2022-2025 completo',
        'F1':        metricas_03a['F1'],
        'AUC_ROC':   metricas_03a['AUC_ROC'],
        'AUC_PR':    metricas_03a['AUC_PR'],
        'Recall':    metricas_03a['Recall'],
        'Bal_Acc':   metricas_03a['Bal_Acc'],
    },
    {
        'Escenario': 'C. Re-entrenado temporalmente',
        'Train': '2015-2021',
        'Test':  '2022-2025 completo',
        'F1':        metricas_temp['F1'],
        'AUC_ROC':   metricas_temp['AUC_ROC'],
        'AUC_PR':    metricas_temp['AUC_PR'],
        'Recall':    metricas_temp['Recall'],
        'Bal_Acc':   metricas_temp['Bal_Acc'],
    },
])

display(tabla_final)
tabla_final.to_csv(OUT_TAB / 'comparativa_temporal_DCSA.csv', index=False)

# Grafico comparativo de metricas
metricas_plot = ['F1','AUC_ROC','AUC_PR','Recall','Bal_Acc']
colores_esc   = [COLORS['neutral'], COLORS['secondary'], COLORS['primary']]

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(metricas_plot))
w = 0.25

for i, (_, row) in enumerate(tabla_final.iterrows()):
    vals = [row[m] for m in metricas_plot]
    bars = ax.bar(x + i*w - w, vals, w,
        color=colores_esc[i], alpha=0.85,
        label=row['Escenario'], edgecolor='white')
    ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=7.5, rotation=90)

ax.set_xticks(x)
ax.set_xticklabels(metricas_plot, fontsize=11)
ax.set_ylabel('Puntuacion')
ax.set_title(
    'Comparativa de escenarios de evaluacion\n'
    f'Modelo base: {mejor_03a} | DCSA')
ax.set_ylim(0, 1.12)
ax.legend(fontsize=8, loc='upper right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(OUT_FIG / 'temporal_comparativa_escenarios.png',
    dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Curvas ROC comparativas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: ROC
if tiene_aleatorio:
    RocCurveDisplay.from_predictions(
        y_test_al, y_prob_al,
        name='Test aleatorio (03a)', color=COLORS['neutral'], ax=axes[0])
RocCurveDisplay.from_predictions(
    y_test_t, y_prob_03a,
    name='Test temporal — modelo 03a', color=COLORS['secondary'], ax=axes[0])
RocCurveDisplay.from_predictions(
    y_test_t, y_prob_temp,
    name='Re-entrenado temporal', color=COLORS['primary'], ax=axes[0])
axes[0].plot([0,1],[0,1],'k--',alpha=0.4)
axes[0].set_title('A. Curvas ROC — 3 escenarios')
axes[0].grid(alpha=0.3)

# Panel B: Precision-Recall
if tiene_aleatorio:
    PrecisionRecallDisplay.from_predictions(
        y_test_al, y_prob_al,
        name='Test aleatorio (03a)', color=COLORS['neutral'], ax=axes[1])
PrecisionRecallDisplay.from_predictions(
    y_test_t, y_prob_03a,
    name='Test temporal — modelo 03a', color=COLORS['secondary'], ax=axes[1])
PrecisionRecallDisplay.from_predictions(
    y_test_t, y_prob_temp,
    name='Re-entrenado temporal', color=COLORS['primary'], ax=axes[1])
axes[1].set_title('B. Curvas Precision-Recall — 3 escenarios')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.suptitle(
    f'Robustez temporal — {mejor_03a}\nDCSA: dengue con signos de alarma',
    fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(OUT_FIG / 'temporal_curvas_roc_pr.png',
    dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## 6. Conclusiones de la validación temporal

### Interpretación de los tres escenarios

**Escenario A (test aleatorio, 03a):** línea base optimista.
La mezcla aleatoria de años en train y test introduce **data leakage temporal**
implícito: el modelo ve datos de 2022–2025 durante el entrenamiento,
lo que infla artificialmente las métricas reportadas en 03a.

**Escenario B (modelo 03a sobre test temporal):** evaluación real de generalización.
Mide cuánto se degrada el modelo entrenado con datos mixtos al enfrentarse
a un período nuevo. Una degradación alta indica dependencia del período de entrenamiento.

**Escenario C (re-entrenamiento temporal):** límite superior alcanzable
con partición temporal estricta. Si supera significativamente al escenario B,
confirma que el modelo puede adaptarse a nuevos regímenes con re-entrenamiento.

### Implicaciones para el despliegue clínico

La ruptura estructural 2021–2022 en la tasa de hospitalización DCSA
(de ~40% a ~87%) tiene dos lecturas:

1. **Cambio de protocolo INS:** el criterio de clasificación DCSA se volvió
   más estricto, por lo que los casos clasificados como DCSA en 2022–2025
   son más graves que los de 2015–2020.
   El modelo entrenado en el régimen antiguo puede subestimar la hospitalización.

2. **Cambio epidemiológico real:** mayor circulación de serotipos con mayor
   virulencia en el período reciente.

**Recomendación:** para despliegue en producción, re-entrenar el modelo
con datos de los últimos 2–3 años (ventana deslizante) y establecer
un monitoreo mensual de las métricas sobre casos recientes.

**Siguiente:** `04_clustering/04_analisis_no_supervisado.ipynb`